##Phase 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

#install BLIP model
!pip install transformers torch -q

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

# Check if GPU available
print("Is GPU available?", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

Device: cuda
Is GPU available? True
Device name: Tesla T4


In [ ]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model = model.to(device)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [ ]:
# Load the clean dataset saved in Phase 1
df = pd.read_csv("/content/drive/MyDrive/Multimodal/multibully_clean.csv")
IMG_DIR = "/content/drive/MyDrive/Multimodal/bully_data/"

# Add image paths
df["image_path"] = df["Img-Name"].apply(lambda x: os.path.join(IMG_DIR, str(x)))

# Add bully label if not already there
if "bully_label" not in df.columns:
    df["bully_label"] = df["Img-Label"].map({"Bully": 1, "Nonbully": 0})

print(f"Dataset loaded : {len(df)} rows")
print(f"Columns        : {df.columns.tolist()}")
print(f"Sample path    : {df['image_path'].iloc[0]}")

Dataset loaded : 5854 rows
Columns        : ['Img-Name', 'Img-Text', 'Img-Label', 'Img-Text-Label', 'Text-Label', 'Sentiment', 'Emotion', 'Sarcasm', 'Harmful-Score', 'Target', 'Doubtful', 'image_path', 'bully_label']
Sample path    : /content/drive/MyDrive/Multimodal/bully_data/0.jpg


In [ ]:
def generate_caption(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=40)
        caption = processor.decode(output[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        return f"ERROR: {str(e)}"

# Test on first 5 rows until we find one that works
print("Testing caption function on first 5 rows...")
print("-" * 50)

for i in range(5):
    test_row = df.iloc[i]
    test_caption = generate_caption(test_row["image_path"])

    print(f"\nRow {i}")
    print(f"  Image path  : {test_row['image_path']}")
    print(f"  File exists : {os.path.exists(test_row['image_path'])}")
    print(f"  Caption     : {test_caption}")

    if test_caption and not test_caption.startswith("ERROR"):
        print("\n  Caption looks good!")
        break
    else:
        print("  Skipping this row...")

Testing caption function on first 5 rows...
--------------------------------------------------

Row 0
  Image path  : /content/drive/MyDrive/Multimodal/bully_data/0.jpg
  File exists : True
  Caption     : a group of women with glasses on their faces

  Caption looks good!


In [ ]:
# Using stratified sampling to check examples

SEED = 42
test_sample_bully = df[df["bully_label"] == 1].sample(6, random_state=SEED)
test_sample_not_bully = df[df["bully_label"] == 0].sample(6, random_state=SEED)
test_sample = pd.concat([test_sample_bully, test_sample_not_bully]).reset_index(drop=True)

print("TESTING BLIP ON 12 SAMPLE IMAGES (6 bully + 6 not-bully)")
print("----------------------------------------------------------")

for _, row in test_sample.iterrows():
    caption = generate_caption(row["image_path"])
    label = "BULLY" if row["bully_label"] == 1 else "NOT BULLY"

    print(f"\n[{label}] {row['Img-Name']}")
    print(f"  Meme text   : {str(row['Img-Text'])[:60]}...")
    print(f"  BLIP caption: {caption}")

TESTING BLIP ON 12 SAMPLE IMAGES (6 bully + 6 not-bully)
----------------------------------------------------------

[BULLY] 2644.jpg
  Meme text   : ye hamari car hai aur ye hum hai aur ye hamari parwi ho rahi...
  BLIP caption: ERROR: [Errno 2] No such file or directory: '/content/drive/MyDrive/Multimodal/bully_data/2644.jpg'

[BULLY] 5713.jpg
  Meme text   : *memes  I share to my crush to impress her* *my crush iq lev...
  BLIP caption: a cartoon character is standing in a room

[BULLY] 2538.jpg
  Meme text   : one again it's proved bad people dont deserve a nose!...
  BLIP caption: a poster with the faces of people in different costumes

[BULLY] 4526.jpg
  Meme text   : *In KBC* *Contestant says 'i quit' just after listening to q...
  BLIP caption: a man with a mustache and a mustache

[BULLY] 4601.jpg
  Meme text   : tharkiyo laar met tapka dena...
  BLIP caption: a woman in a bathing suit is throwing water

[BULLY] 5946.jpg
  Meme text   : Electric Car Charging Point , Running On

In [ ]:
import time

# Check if I already have a partial save from a previous run
SAVE_PATH = "multibully_with_captions.csv"
SAVE_EVERY = 500  # save progress after every 500 images

if os.path.exists(SAVE_PATH):

    df_saved = pd.read_csv(SAVE_PATH)
    already_done = df_saved[df_saved["blip_caption"].notna()]["Img-Name"].tolist()
    print(f"Found existing save — {len(already_done)} captions already done")

else:
    # Starting fresh
    df["blip_caption"] = None
    already_done = []

print(f"\nTotal images to caption: {len(df)}")
print(f"Already done           : {len(already_done)}")
print(f"Remaining              : {len(df) - len(already_done)}")
print("\nStarting... (saving every 500 images)")
print("-" * 50)

start_time = time.time()
count = 0
errors = 0

for idx, row in df.iterrows():

    # Skip if already done
    if row["Img-Name"] in already_done:
        continue

    # Generate caption
    caption = generate_caption(row["image_path"])
    df.at[idx, "blip_caption"] = caption

    # Count errors
    if caption.startswith("ERROR"):
        errors += 1

    count += 1

    # Save progress every 500 images
    if count % SAVE_EVERY == 0:
        df.to_csv(SAVE_PATH, index=False)
        elapsed = time.time() - start_time
        remaining = (elapsed / count) * (len(df) - len(already_done) - count)
        print(f"  Progress: {count + len(already_done)}/{len(df)} images done | "
              f"Errors: {errors} | "
              f"Est. time left: {remaining/60:.0f} mins")

    # Print update every 100 images
    elif count % 100 == 0:
        print(f"  {count + len(already_done)}/{len(df)} done...")

# Final save when completely done
df.to_csv(SAVE_PATH, index=False)

total_time = time.time() - start_time
print(f"\nDONE!")
print(f"Total images captioned : {count}")
print(f"Errors                 : {errors}")
print(f"Total time             : {total_time/60:.1f} minutes")
print(f"Saved to               : {SAVE_PATH}")


Total images to caption: 5854
Already done           : 0
Remaining              : 5854

Starting... (saving every 500 images)
--------------------------------------------------
  100/5854 done...
  200/5854 done...
  300/5854 done...
  400/5854 done...
  Progress: 500/5854 images done | Errors: 7 | Est. time left: 48 mins
  600/5854 done...
  700/5854 done...
  800/5854 done...
  900/5854 done...
  Progress: 1000/5854 images done | Errors: 10 | Est. time left: 41 mins
  1100/5854 done...
  1200/5854 done...
  1300/5854 done...
  1400/5854 done...
  Progress: 1500/5854 images done | Errors: 15 | Est. time left: 38 mins
  1600/5854 done...
  1700/5854 done...
  1800/5854 done...
  1900/5854 done...
  Progress: 2000/5854 images done | Errors: 19 | Est. time left: 34 mins
  2100/5854 done...
  2200/5854 done...
  2300/5854 done...
  2400/5854 done...
  Progress: 2500/5854 images done | Errors: 22 | Est. time left: 29 mins
  2600/5854 done...
  2700/5854 done...
  2800/5854 done...
  2900/